# Report MS COCO classification challenge

**IAV — INSA Lyon**  
Tayeb and Paul

Repository: <https://github.com/proche0/IAV_MS_COCO>

Each image may contain several of the 80 MS COCO classes. The ranking metric is the F1-score. Training relies solely on ImageNet weights, using the 65,000 annotated images provided for the course. The official test set (4,952 images) is reserved for the submission JSON file.

The F1-score is chosen over accuracy because it is not skewed when one class is much more frequent than another, ensuring the model performs well across all categories rather than just the most populous one.

Here are the key formulas for this challenge:

$$
P = \frac{TP}{TP + FP},
\quad
R = \frac{TP}{TP + FN}
$$

$$
F_1 = \frac{2PR}{P+R}
$$

P : Precision

R : Recall


## Contents

1. [Use of AI](#use-of-ai)
2. [Scientific approach](#scientific-approach)
3. [Ranking of architectures](#ranking-of-architectures)
4. [Bibliography](#bibliography)


<a id="use-of-ai"></a>

## 1. Use of AI

An AI coding assistant was used while building the project.

**What the assistant produced**

- A first draft of the shared library `src/coco_mlc`: datasets, the stratified split, the losses, a copy of the F1 score, the training loop, and the threshold calibration.

**What we decided, and what we check**

- The protocol in section 2: the 70/15/15 split, asymmetric loss, the two training stages, and per-class thresholds fitted on validation only.
- The shortlist in section 3, including the course baselines (VGG16, ResNet18) and the heavier models that fit on the GPU.
- Every score that will appear in the ranking. Those scores will be copied from the experiment outputs and checked against the printed metrics. They will not be estimated by the assistant.

Generated code is kept only after it has been read. A reported number comes from a full run (`FULL_TRAIN = True`). A short run (512 images, one epoch) only checks that the notebook executes.


<a id="scientific-approach"></a>

## 2. Scientific approach

### 2.1 Question

Which ImageNet-pretrained network, trained using the same transfer learning protocol, achieves the highest F1 score on images for which it was not fine-tuned?

To address this question, the main approach was to test several architectures based on available hardware. The loss function, data split, augmentation scheme, optimizer, and number of epochs remain constant. Batch size is the variable parameter, as it is constrained by GPU memory; it is specified for each model and factored into the cost.

### 2.2 Why the metric changes the objective

The F1 score weights each class by the inverse of its frequency, then takes the harmonic mean of the weighted precision and the weighted recall. The formula is in the introduction. On the 65,000 labeled images, rare classes carry most of the score: `hair drier` (102 images) is about 13.6% of the score, `toaster` (117) about 11.9%, and the ten rarest classes about 43.8%. `person` (35,494 images) is about 0.04%.

A score dominated by frequent classes would miss the leaderboard. Training follows from that fact: a loss that does not let the many negative labels drown the rare positives, and one decision threshold per class, fitted on validation.

Training curves plot the error `100 × (1 − F1 score)` at a fixed threshold of 0.5, so runs can be compared before calibration.

### 2.3 Data

| Set | Role | Size |
| --- | --- | --- |
| Labeled images | the only images with class files | 65,000 |
| Train | weight updates | 70% |
| Validation | reading the curves, threshold calibration, model selection | 15% |
| Local test | generalization, read once after the protocol is frozen | 15% |
| Official test | leaderboard JSON, no labels on our side | 4,952 |

The three labeled subsets are disjoint. The split is iterative stratification (Sechidis et al., 2011) with seed 42, cached in `outputs/split_three_stratified_0.7_0.15_0.15_42.json`, and shared by every architecture. A plain random split can leave a rare class out of validation, which makes both the metric and the thresholds unstable.

Images already have a long side of 224 and varying aspect ratios. We pad to a square, then resize to 224×224. A center crop would drop border objects whose label would still be positive. Pixels are normalized with the ImageNet mean and standard deviation, matching the pretrained backbones.

### 2.4 Training protocol

Every notebook follows the same two stages. This pair of stages is one iteration.

**Stage A — frozen backbone.** ImageNet weights, classification layer replaced by 80 logits. The backbone is frozen. Only the new head is trained, for 5 epochs, AdamW, learning rate `1e-3`, no weight decay, no dropout. Augmentation is a horizontal flip. The head bias is initialized to the log-odds of the class prevalence on the **train** subset, so the network does not start from probability 0.5 on labels that occur a few percent of the time.

**Stage B — fine-tuning.** The backbone is unfrozen. 8 epochs, AdamW, learning rate `1e-4`, weight decay `1e-4`, dropout 0.3 on the head. Augmentation adds a light affine transform (rotation, translation, scale) and color jitter. The learning rate follows a cosine schedule over the mini-batches of the stage. Mixed precision is used on CUDA.

The loss is the asymmetric loss (Ben-Baruch et al., 2021): stronger focusing on negatives (`gamma_neg = 4`, `gamma_pos = 0`) and a clip of 0.05 on easy negatives. With only a few positive labels out of 80, a plain binary cross-entropy spends most of its gradient on negatives.

The network returns logits. The sigmoid is applied only when scores become metrics or a JSON file.

### 2.5 Thresholds and the local test

After fine-tuning, probabilities are collected on the validation set. One threshold per class is chosen by coordinate ascent on the F1 score, on a grid from 0.02 to 0.90. A single threshold of 0.5 is a weak default: rare classes need a lower threshold to gain recall, and that recall is heavily weighted.

The local test is scored after this choice, once, at threshold 0.5 and with the calibrated thresholds. It is not used to change the loss, the epochs, the learning rates, or the thresholds. If the local test is much worse than validation, the validation set was overfit, and we do not retune on the local test.

The official test is used only to write `submissions/exp_<model>.json`, with the thresholds chosen on validation.

### 2.6 How a model will be kept

1. Keep a run only if both stages finished on the full labeled set.
2. Rank finished runs by the **calibrated F1 score on the local test**.
3. Read the calibrated validation F1 score as a consistency check. The seed is shared, so this is not a second test set.
4. Break a near-tie with cost: parameter count, GFLOPs at 224×224, and the batch size that fit in memory.
5. Submit one JSON, under one group name, for the retained model.

Bias and variance are read from the train/validation error gap at the last epoch, with the cases already coded in the experiment notebooks: underfitting, overfitting, both, or a jump from validation to the local test.

### 2.7 Architectures and the available GPU

Every run uses one GPU, an NVIDIA GeForce RTX 4060 Ti, with mixed precision and images of 224×224. The batch size is the only setting changed so that the forward and backward activations fit in that memory:

- Light models (ShuffleNet V2 ×1.0, MobileNetV3-Large, EfficientNet-B0, ResNet18) run at batch 64 to 128.
- Mid-size models (ResNet50, ConvNeXt-Tiny, Swin-T) run at batch 64 to 96.
- Heavier models (EfficientNet-B4, EfficientNetV2-S) run at batch 16.
- MaxViT-T runs at batch 8. MaxViT-T is the most expensive recent backbone that still fits.


<a id="ranking-of-architectures"></a>

## 3. Ranking of architectures

### 3.1 Experiment 1

Experiment 1 is the first iteration: every notebook in `notebooks/experiments/1/`, trained once with the protocol of section 2. The F1 score below is the calibrated F1 score on the local test, read from `outputs/notebooks/<architecture>/generalization.csv`. Parameter counts, GFLOPs and ImageNet top-1 come from the torchvision weights table at 224×224, before our head replacement. The batch size is the one set in each notebook.


![Rank 1](results/ranks/rank1.png)

#### Validation curves

The two figures below show every architecture that finished experiment 1. They are drawn by [`results/curves/curves.ipynb`](results/curves/curves.ipynb). Each line is the validation curve. Epochs 1–5 are the frozen-backbone stage. Epochs 6–13 are fine-tuning. The dashed line marks the change of stage.

Error is `100 × (1 − F1 score)` at a threshold of 0.5, before calibration. Loss is the asymmetric loss on the validation set.


![Validation error — experiment 1](results/leaderboard/experiment_1_validation_error.png)

![Validation loss — experiment 1](results/leaderboard/experiment_1_validation_loss.png)


### 3.2 Experiment 2

The two best architectures of experiment 1 are trained again, as a second iteration. On the F1 score of experiment 1 they are MaxViT-T and ConvNeXt-Tiny. Their notebooks are in `notebooks/experiments/2/`.

The second iteration keeps the same split, the same asymmetric loss, and the same two stages: a frozen backbone, then fine-tuning. Stage A trains the head for 10 epochs. Stage B fine-tunes the backbone for 10 epochs. Images are loaded in the main process (`num_workers = 0`), and mixed precision is used on CUDA.

No F1 score is reported here until those three notebooks have written `outputs/notebooks/2/<architecture>/generalization.csv`. The ranking and the validation curves are inserted here, from captures of [`results/ranks/ranking.ipynb`](results/ranks/ranking.ipynb) and [`results/curves/curves.ipynb`](results/curves/curves.ipynb), saved in `results/leaderboard/`.

![Rank 2](results/ranks/rank2.png)

#### Validation curves

![Curve 1](results/curves/curve1.png)

![Curve 2](results/curves/curve2.png)


<a id="bibliography"></a>

## 4. Bibliography

- T.-Y. Lin, M. Maire, S. Belongie, J. Hays, P. Perona, D. Ramanan, P. Dollár, and C. L. Zitnick. *Microsoft COCO: Common Objects in Context.* ECCV, 2014.
- R. Kéchichian. *The MS COCO classification challenge.* IAV, INSA Lyon. 
- Torchvision models and weights. <https://docs.pytorch.org/vision/main/models.html>. Source of the parameter counts, GFLOPs, and ImageNet top-1 figures in section 3.
- E. Ben-Baruch, T. Ridnik, N. Zamir, et al. *Asymmetric Loss For Multi-Label Classification.* ICCV, 2021.
- K. Sechidis, G. Tsoumakas, and I. Vlahavas. *On the Stratification of Multi-label Data.* ECML PKDD, 2011.
- T.-Y. Lin, P. Goyal, R. Girshick, K. He, and P. Dollár. *Focal Loss for Dense Object Detection.* ICCV, 2017. The head bias is initialized from the class prevalence, following the same idea.
- I. Loshchilov and F. Hutter. *Decoupled Weight Decay Regularization.* ICLR, 2019.
